# RAG & Knowledge Systems: Chunking Strategies

In the previous notebook, we used a small list of facts. In the real world, documents (PDFs, Wiki pages) are thousands of lines long. We must **Chunk** the data.

## 1. Why Chunking Matters?
**Chunking** is the process of breaking a document into smaller pieces. 
- **Precision**: Smaller chunks allow the retriever to find the exact paragraph needed.
- **Context Window**: Most LLMs have a limit on how much text they can process in one go.
- **Embeddings**: An embedding for an entire book is less useful than an embedding for a specific chapter.

---

## 2. Environment Setup
We need `langchain_experimental` for advanced semantic splitting and `langchain_huggingface` for local embeddings.

In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

load_dotenv()
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

print("Chunking environment ready!")

/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunking environment ready!


## 3. Fixed-Size Chunking
The simplest form of chunking. It splits text at exactly N characters, regardless of content.

In [2]:
text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. 
AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
The term 'artificial intelligence' had previously been used to describe machines that mimic and display human cognitive skills such as learning and problem-solving.
"""

splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=50,
    chunk_overlap=5
)

chunks = splitter.split_text(text)
print(f"Fixed-size Chunks (Total: {len(chunks)}):")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Fixed-size Chunks (Total: 12):
Chunk 1: Artificial intelligence (AI) is intelligence
Chunk 2: demonstrated by machines, as opposed to the
Chunk 3: the natural intelligence displayed by animals
Chunk 4: including humans. 
AI research has been defined as
Chunk 5: as the field of study of intelligent agents, which
Chunk 6: which refers to any system that perceives its
Chunk 7: its environment and takes actions that maximize
Chunk 8: its chance of achieving its goals.
The term
Chunk 9: term 'artificial intelligence' had previously been
Chunk 10: been used to describe machines that mimic and
Chunk 11: and display human cognitive skills such as
Chunk 12: as learning and problem-solving.


## 4. Recursive Character Splitting (The Gold Standard)
Instead of splitting arbitrarily, it tries to split at paragraphs (`\n\n`), then sentences (`\n`), then spaces (` `). This keeps semantically related text together.

In [3]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

r_chunks = recursive_splitter.split_text(text)
print(f"Recursive Chunks (Total: {len(r_chunks)}):")
for i, chunk in enumerate(r_chunks):
    print(f"Chunk {i+1}: {chunk}")

Recursive Chunks (Total: 7):
Chunk 1: Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural
Chunk 2: to the natural intelligence displayed by animals including humans.
Chunk 3: AI research has been defined as the field of study of intelligent agents, which refers to any
Chunk 4: which refers to any system that perceives its environment and takes actions that maximize its
Chunk 5: that maximize its chance of achieving its goals.
Chunk 6: The term 'artificial intelligence' had previously been used to describe machines that mimic and
Chunk 7: that mimic and display human cognitive skills such as learning and problem-solving.


## 5. Semantic Chunking (Adaptive Splitting)
This is the most advanced technique. It uses embeddings to calculate the "distance" between sentences. If two sentences are semantically different, it creates a new chunk.

In [4]:
semantic_splitter = SemanticChunker(embeddings)

# Sample text with two different topics
long_text = """
The recipe for a classic chocolate cake requires cocoa powder, flour, and sugar. 
Make sure to whisk the eggs until fluffy before adding them to the batter.
Quantum computing utilizes qubits to perform calculations at speeds impossible for classical computers.
Superposition and entanglement are core principles of quantum mechanics that power these machines.
"""

s_chunks = semantic_splitter.create_documents([long_text])
print(f"Semantic Chunks (Total: {len(s_chunks)}):")
for i, chunk in enumerate(s_chunks):
    print(f"Chunk {i+1}:\n{chunk.page_content}\n")

Semantic Chunks (Total: 2):
Chunk 1:

The recipe for a classic chocolate cake requires cocoa powder, flour, and sugar. Make sure to whisk the eggs until fluffy before adding them to the batter. Quantum computing utilizes qubits to perform calculations at speeds impossible for classical computers.

Chunk 2:
Superposition and entanglement are core principles of quantum mechanics that power these machines. 



## 6. Similarity Analysis between Chunks
Let's visualize how the semantic splitter decided where to break.

In [5]:
import numpy as np

sentences = [
    "The cat sat on the mat.",
    "The feline was resting on the rug.",
    "The airplane flew high in the sky."
]

v_cat1 = np.array(embeddings.embed_query(sentences[0]))
v_cat2 = np.array(embeddings.embed_query(sentences[1]))
v_plane = np.array(embeddings.embed_query(sentences[2]))

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Similarity (Cat 1 vs Cat 2): {cosine_similarity(v_cat1, v_cat2):.4f}")
print(f"Similarity (Cat 1 vs Plane): {cosine_similarity(v_cat1, v_plane):.4f}")

Similarity (Cat 1 vs Cat 2): 0.5630
Similarity (Cat 1 vs Plane): 0.0961


## 7. Importance of Chunk Overlap
**Overlap** ensures that context isn't lost at the split point. If a sentence is cut in half, the overlap provides the rest of the sentence to both chunks.

In [6]:
overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=20
)
test_text = "Learning Agentic AI is a journey that requires persistence and practice."
o_chunks = overlap_splitter.split_text(test_text)
for i, chunk in enumerate(o_chunks):
    print(f"Chunk {i+1}: {chunk}")

Chunk 1: Learning Agentic AI is a journey that requires
Chunk 2: that requires persistence and practice.


## 8. Metadata Enrichment
When you chunk a document, you should store metadata (Page number, Source, Timestamp) to help the agent provide citations.

In [7]:
from langchain_core.documents import Document

doc = Document(
    page_content="This is a secret fact about agents.",
    metadata={"source": "agent_handbook.pdf", "page": 42}
)
print(f"Metadata: {doc.metadata}")

Metadata: {'source': 'agent_handbook.pdf', 'page': 42}


## 9. Dealing with Special Formats (Markdown/HTML)
LangChain provides specialized splitters that understand syntax (e.g., Markdown headers).

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_document = "# Title\n\n## Section A\nText A\n\n## Section B\nText B"
headers_to_split_on = [("#", "Header 1"), ("##", "Header 2")]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
print(f"Markdown Header Split 1 Content: {md_header_splits[0].page_content}")
print(f"Markdown Header Split 1 Metadata: {md_header_splits[0].metadata}")

Markdown Header Split 1 Content: Text A
Markdown Header Split 1 Metadata: {'Header 1': 'Title', 'Header 2': 'Section A'}


## 10. Summary & Pro-Tips
1. **Never use Fixed-Size for Production**: It cuts sentences in half, ruining embeddings.
2. **Use Semantic Chunking for Complex Data**: When topic shifts are unpredictable, semantic chunking is king.
3. **Overlap is Mandatory**: A 10-20% overlap is usually the sweet spot for maintaining continuity.
4. **Local Embeddings**: We are using Hugging Face's `all-MiniLM-L6-v2` which runs entirely on your CPU/GPU.

---